# Multimodal Fusion for Turn-Taking Prediction

### CSCI-535 POC — 3-class horizon-curve turn-taking classifier

We adapt the practicum's audiovisual-fusion pipeline to a **stride-based, horizon-centric turn-taking prediction task** on the Meta Seamless Interaction corpus. At each sample point $t$ during a perspective participant's turn, we predict the 3-class label of the forward horizon $[t, t+\tau]$:

- **HOLD (0)** — current speaker keeps the floor
- **YIELD (1)** — current speaker yields to partner
- **BACKCHANNEL (2)** — partner emits a short filler, speaker keeps the floor

Samples that would have been **INTERRUPT / FAILED / LAPSE** under the exclusion sub-types (spec §11.3) are labeled but filtered out of training.

We use two pre-extracted feature sets ([build_labeled_windows_from_manifest.py](../scripts/build_labeled_windows_from_manifest.py), see spec §11):

- **OpenFace** (visual, 20-dim): 17 AU intensities + 3 head-pose angles, 30 fps → 60 frames per 2 s window
- **WavLM-base+** (audio, 768-dim): penultimate transformer layer, 10 Hz pooled → 20 frames per 2 s window

### Experiments (spec §11.7 rows 1, 3, 6, 9, 9-shuffled)

| # | Run name | Visual stream | Audio stream | Models |
|---|---|---|---|---|
| 1 | `row1_dyadic_cross_pair` | listener's face | speaker's audio | unimodal × 2 + Early + Neural-Concat + Late |
| 3 | `row3_unimodal_audio` | — | speaker's audio | unimodal audio only |
| 6 | `row6_permuted_dyad_control` | listener's face *(shuffled at test)* | speaker's audio | reuses row 1 weights |
| 9 | `row9_mmvap_full_dyadic` | concat(speaker, listener) face | concat(speaker, listener) audio | unimodal × 2 + Early + Neural-Concat + Late |
| 9′ | `row9_shuffled_control` | concat with listener half shuffled at test | concat with listener half shuffled at test | reuses row 9 weights |

Training is done once per experiment at **τ = 400 ms** (Inoue et al. 2024 operational horizon). The same trained model is re-evaluated across τ ∈ {100, 200, 400, 500, 800, 1600} ms to build the **horizon curve** (spec §6.2.e).

All fusion strategies are unchanged from the practicum:

| # | Strategy | Where | Key Idea |
|---|---|---|---|
| 1 | **Early Fusion** | Input | Concatenate features before the model |
| 2 | **Late Fusion** | Output | Combine predictions from separate models |
| 3 | **Neural Concatenation** | Representation | Join learned hidden states mid-network |

## Notebook overview

This notebook reuses the practicum's PyTorch fusion code with minimum edits. The core PyTorch mechanics are unchanged — we're just retargeting the task, labels, and data paths.

### PyTorch concepts used
| Concept | Where used |
|---------|------------|
| `Dataset` / `DataLoader` | All sections |
| `nn.Module` subclassing | Sections 3–7 |
| `model.train ` / `model.eval ` | All training loops |
| `optimizer.zero_grad ` / `loss.backward ` / `optimizer.step ` | All training loops |
| `torch.no_grad ` | Evaluation |
| `.to(device)` for GPU | Throughout |
| Manual early stopping | Training helper |

### What's new vs. the practicum
- 3 classes (HOLD / YIELD / BACKCHANNEL) instead of 6 emotions
- Label lookup from `labels_tau_XXXX.json` instead of filename field `f.split('_')[2]`
- `MAX_LEN_V = 60` (was 16), `MAX_LEN_A = 20` (was 40) — reflects 2-s window at 30 fps / 10 Hz
- `ConcatFEATDataset` wrapper (added in Cell 10) for dyadic-concat experiments
- τ-sweep evaluation loop at the end to build horizon curves
- Permuted-dyad control as a named validity check (spec §8.2)

---
## Section 0: Setup


In [ ]:
!pip install -q tqdm seaborn scikit-learn

In [ ]:
import os, json
from pathlib import Path

# ── Project-root resolution ────────────────────────────────────────────
# Notebook lives at .../turn_taking_analysis/scripts/multimodal_fusion_for_turns.ipynb
# model_input/ is one level up under turn_taking_analysis/.
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = (NOTEBOOK_DIR.parent
                if NOTEBOOK_DIR.name == 'scripts'
                else NOTEBOOK_DIR)
MODEL_INPUT = PROJECT_ROOT / 'model_input'
assert MODEL_INPUT.exists(), f"model_input/ not found at {MODEL_INPUT}"
print(f"MODEL_INPUT: {MODEL_INPUT}")

# ── Per-τ labels ───────────────────────────────────────────────────────
# Each JSON is {basename: class_int 0..5}. Training uses τ=400 labels.
# The horizon-curve eval (later section) re-reads each τ's labels.
TAU_GRID_MS = (100, 200, 400, 500, 800, 1600)
TRAIN_TAU_MS = 400

LABELS_BY_TAU = {
    tau_ms: json.load(open(MODEL_INPUT / 'labels' / f'labels_tau_{tau_ms:04d}.json'))
    for tau_ms in TAU_GRID_MS
}
TRAIN_LABELS = LABELS_BY_TAU[TRAIN_TAU_MS]
print(f"Loaded τ-labels for: {sorted(LABELS_BY_TAU.keys())} ms (train uses {TRAIN_TAU_MS} ms)")

# ── Ablation experiments (spec §11.7 rows 1, 3, 6, 9, 9-shuffled) ──────
# feat_path_v / feat_path_a are LISTS of directories. Length-1 → plain
# SequenceDataset / BimodalDataset. Length-2 → ConcatFEATDataset (concat
# along feature axis at __getitem__ time, spec §11.8).
EXPERIMENTS = {
    'row1_dyadic_cross_pair': dict(
        description='Dyadic cross-pair: listener face + speaker audio (primary)',
        feat_path_v=[MODEL_INPUT / 'openface' / 'listener'],
        feat_path_a=[MODEL_INPUT / 'wavlm' / 'speaker'],
        feat_v=20, feat_a=768,
        models=('unimodal_v', 'unimodal_a', 'early_fusion',
                'neural_concat', 'late_fusion'),
        permute_v_at_test=False,
        reuse_weights_from=None,
    ),
    'row3_unimodal_audio': dict(
        description='Unimodal audio-only baseline (no visual)',
        feat_path_v=None,
        feat_path_a=[MODEL_INPUT / 'wavlm' / 'speaker'],
        feat_v=None, feat_a=768,
        models=('unimodal_a',),
        permute_v_at_test=False,
        reuse_weights_from=None,
    ),
    'row6_permuted_dyad_control': dict(
        description='Permuted-dyad control on row 1 (listener face shuffled at test)',
        feat_path_v=[MODEL_INPUT / 'openface' / 'listener'],
        feat_path_a=[MODEL_INPUT / 'wavlm' / 'speaker'],
        feat_v=20, feat_a=768,
        models=('early_fusion', 'neural_concat', 'late_fusion'),
        permute_v_at_test=True,
        reuse_weights_from='row1_dyadic_cross_pair',  # no retraining
    ),
    'row9_mmvap_full_dyadic': dict(
        description='MM-VAP-style full dyadic: both-face + both-audio',
        feat_path_v=[MODEL_INPUT / 'openface' / 'speaker',
                     MODEL_INPUT / 'openface' / 'listener'],
        feat_path_a=[MODEL_INPUT / 'wavlm' / 'speaker',
                     MODEL_INPUT / 'wavlm' / 'listener'],
        feat_v=40, feat_a=1536,
        models=('unimodal_v', 'unimodal_a', 'early_fusion',
                'neural_concat', 'late_fusion'),
        permute_v_at_test=False,
        reuse_weights_from=None,
    ),
    'row9_shuffled_control': dict(
        description='Permuted-dyad control on row 9 (listener half shuffled at test)',
        feat_path_v=[MODEL_INPUT / 'openface' / 'speaker',
                     MODEL_INPUT / 'openface' / 'listener'],
        feat_path_a=[MODEL_INPUT / 'wavlm' / 'speaker',
                     MODEL_INPUT / 'wavlm' / 'listener'],
        feat_v=40, feat_a=1536,
        models=('early_fusion', 'neural_concat', 'late_fusion'),
        permute_v_at_test=True,
        reuse_weights_from='row9_mmvap_full_dyadic',
    ),
}
print(f"Planned experiments: {list(EXPERIMENTS.keys())}")

# ── Split file lists (shared across all experiments) ───────────────────
# Every (modality, role) directory contains the same set of basenames.
# Enumerate once from openface/speaker and filter to samples whose τ=400
# label is a training class {0, 1, 2}. EXCLUDE samples (INTERRUPT/FAILED/
# LAPSE) are kept in labels JSONs for stats but removed from these lists.
ENUM_DIR = MODEL_INPUT / 'openface' / 'speaker'
splits = {}
print("\nBuilding splits (filtered to {HOLD, YIELD, BACKCHANNEL} at τ=400):")
for split in ('train', 'val', 'test'):
    all_files = sorted(f for f in os.listdir(ENUM_DIR / split) if f.endswith('.npy'))
    filtered = [f for f in all_files
                if TRAIN_LABELS.get(f[:-4], 999) in (0, 1, 2)]
    splits[split] = filtered
    n_drop = len(all_files) - len(filtered)
    print(f"  {split:6s}: {len(filtered):5,d} / {len(all_files):5,d} samples kept "
          f"({n_drop:,} dropped as INTERRUPT/FAILED/LAPSE)")


In [ ]:
import numpy as np
import random, copy
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import (accuracy_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device ─────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

# ── Constants (turn-taking POC, see spec §6.2 and §11.5) ───────────────
FEAT_V        = 20    # OpenFace: 17 AU_r + 3 pose_R (§11.5)
FEAT_A        = 768   # WavLM-base+ penultimate (spec §1 audio encoder note)
MAX_LEN_V     = 60    # 2 s × 30 fps OpenFace
MAX_LEN_A     = 20    # 2 s × 10 Hz WavLM
MAX_LEN_EARLY = 20    # early fusion: OpenFace is pooled to 10 Hz first (Cell 10)
NUM_CLASSES   = 3     # {HOLD, YIELD, BACKCHANNEL}
GRU_UNITS     = 64
EPOCHS        = 30
BATCH_SIZE    = 32
LR            = 1e-3

# labels_dict is REFERENCE ONLY — not consumed by any Dataset. The
# Datasets look up integer class IDs directly from the labels JSONs
# (which already store ints). Kept here for human-readability of class
labels_dict = {'HOLD': 0, 'YIELD': 1, 'BACKCHANNEL': 2}
label_names = ['HOLD', 'YIELD', 'BACKCHANNEL']

# results_table stores rich metrics per method:
#   {method: {'acc': float, 'macro_f1': float, 'f1_per_class': [H, Y, B]}}
# So the τ-sweep plot can pull per-class F1 curves.
results_table = {}


---
## Section 1: Feature Exploration

**OpenFace features (20 dimensions):**
AU01_r AU02_r AU04_r AU05_r AU06_r AU07_r AU09_r AU10_r
AU12_r AU14_r AU15_r AU17_r AU20_r AU23_r AU25_r AU26_r
AU45_r pose_Rx pose_Ry pose_Rz

See the [FACS reference](https://www.cs.cmu.edu/~face/facs.htm) for action-unit definitions. At 30 fps with a 2-s window this yields **60 frames per sample**.

**WavLM-base+ features (768-dim):** 
Penultimate transformer layer (layer −2) of [WavLM-base+](https://arxiv.org/abs/2110.13900) (Chen et al. 2022), the SUPERB paralinguistic leader. Originally at 50 Hz (20-ms hop); temporally pooled ×5 → 10 Hz effective rate → **20 frames per sample**.

In [ ]:
# Class balance per split at the training τ (natural rate; EXCLUDE classes dropped in Cell 4)
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=False)
for ax, split in zip(axes, ('train', 'val', 'test')):
    classes = [TRAIN_LABELS[f[:-4]] for f in splits[split]]
    counts = Counter(classes)
    bars = [counts.get(c, 0) for c in (0, 1, 2)]
    ax.bar(label_names, bars, edgecolor='black',
           color=['steelblue', 'tomato', 'seagreen'])
    for n, rect in zip(bars, ax.patches):
        ax.text(rect.get_x() + rect.get_width() / 2,
                rect.get_height() + max(bars) * 0.02,
                f'{n:,}', ha='center', fontsize=9)
    ax.set_title(f'{split} (n={sum(bars):,})')
    ax.tick_params(axis='x', rotation=15)
axes[0].set_ylabel('Count')
plt.suptitle(f'Class balance at τ={TRAIN_TAU_MS} ms '
             f'(all splits at natural rate; imbalance handled via class-weighted loss)')
plt.tight_layout()
plt.show()


> **Note on class balance.** All splits are emitted at natural rate — no HOLD-subsampling. Class imbalance is handled at training time via inverse-frequency weights in `nn.CrossEntropyLoss` (see Cell 14). HOLD dominates the natural distribution; YIELD and BC are the minority classes. Per-class F1 (not accuracy) is the primary metric because it is not inflated by HOLD correctness. The τ-sweep later shows how per-class F1 degrades across horizon.

---
## Section 2: PyTorch Data Loading

In PyTorch, data loading is split across two classes:

| Class | Responsibility |
|-------|----------------|
| `torch.utils.data.Dataset` | Knows how to load and transform **one** sample |
| `torch.utils.data.DataLoader` | Wraps a Dataset; handles batching, shuffling, and parallel workers |

Your `Dataset` must implement:
- `__len__(self)` → total number of samples
- `__getitem__(self, idx)` → the sample at index `idx` as tensors

### Padding strategy
Each `.npy` file already has fixed shape (`MAX_LEN_V` × `FEAT_V` or `MAX_LEN_A` × `FEAT_A`) — the window-building script zero-pads at emission time. We still apply clip/pad defensively in the loaders so shape mismatches surface as loud errors rather than silent corruption.

### Dyadic-concat convention (rows 7–10)
For experiments that feed both speaker + listener features into a single modality slot, `SequenceDataset` / `BimodalDataset` accept a **list of 2 paths** per slot. Convention: **speaker path first, listener path second**. Concatenation happens on the feature axis at `__getitem__` time — no `both/` directory is pre-materialized on disk (spec §11.8).

### Permuted-dyad control (rows 6 and 9-shuffled)
Setting `permute_listener_v=True` or `permute_listener_a=True` on `BimodalDataset` shuffles the listener-stream file-to-sample assignment within the loader. For length-1 path lists (row 6's visual = listener-only), the single path IS the listener → the shuffle applies to the primary file list. For length-2 path lists (row 9-shuffled), the shuffle applies to the secondary file assignment only. When BOTH modality slots are permuted simultaneously, the **same permutation σ** is applied to both — so the "fake listener" whose face and voice we paste in comes from the same randomly-mismatched dyad across modalities. Spec §8.2.

In [ ]:
# ── SequenceDataset: single-modality loader ────────────────────────────
class SequenceDataset(Dataset):
    """Single-modality sequential feature dataset.

    Args
    ----
    path_list : list of 1 or 2 directories. Length-1 loads a single .npy;
        length-2 loads from both and concatenates on the feature axis
        (dyadic-concat per spec §11.8). Convention for length-2:
        path_list[0] = speaker, path_list[1] = listener.
    file_list : list of basenames (with .npy suffix).
    labels_json : dict {basename_without_ext: class_int} at the τ being
        evaluated.
    feat_total : total feature dim AFTER concat (e.g. 20 for length-1
        OpenFace, 40 for length-2 OpenFace). Per-path dim is
        feat_total // len(path_list).
    target_size : fixed sequence length (pad / clip to this).
    permute_listener : if True, shuffle the listener file assignment.
        For length-1 paths the primary file IS the listener → shuffle
        primary. For length-2 paths, shuffle secondary only.
    rng : optional np.random.Generator for reproducibility.
    """
    def __init__(self, path_list, file_list, labels_json,
                 feat_total, target_size,
                 permute_listener=False, rng=None):
        self.path_list = [str(p) for p in path_list]
        self.feat_total = feat_total
        self.target_size = target_size
        self.files_primary  = list(sorted(file_list))
        self.files_listener = list(self.files_primary)
        self.labels = [labels_json[Path(f).stem] for f in self.files_primary]

        rng = rng or np.random.default_rng(42)
        if permute_listener:
            if len(path_list) == 1:
                rng.shuffle(self.files_primary)
            else:
                rng.shuffle(self.files_listener)

    def _pad(self, x, feat_dim):
        T = x.shape[0]
        if T > self.target_size:
            x = x[:self.target_size]
        elif T < self.target_size:
            x = np.vstack([x, np.zeros((self.target_size - T, feat_dim), np.float32)])
        return x

    def __len__(self):
        return len(self.files_primary)

    def __getitem__(self, idx):
        if len(self.path_list) == 1:
            x = np.load(os.path.join(self.path_list[0],
                                     self.files_primary[idx])).astype(np.float32)
            x = self._pad(x, self.feat_total)
        else:
            per = self.feat_total // 2
            xp = np.load(os.path.join(self.path_list[0],
                                      self.files_primary[idx])).astype(np.float32)
            xs = np.load(os.path.join(self.path_list[1],
                                      self.files_listener[idx])).astype(np.float32)
            xp = self._pad(xp, per)
            xs = self._pad(xs, per)
            x  = np.concatenate([xp, xs], axis=-1)
        return torch.tensor(x), self.labels[idx]


# ── BimodalDataset: paired visual + audio loader ──────────────────────
class BimodalDataset(Dataset):
    """Two-modality dataset; yields (x_visual, x_audio, label).

    Each modality slot takes a path_list of length 1 or 2 (see
    SequenceDataset). Length-2 slots concatenate speaker + listener at
    the feature axis at __getitem__ time.

    permute_listener_v / permute_listener_a : independently shuffle each
        modality's listener file assignment at loader-build time. When
        BOTH are True, the SAME permutation σ is applied to both — the
        "fake listener" across face and voice comes from the same
        randomly-mismatched dyad. Spec §8.2.
    """
    def __init__(self, path_list_v, path_list_a, file_list, labels_json,
                 feat_v, feat_a, target_size_v, target_size_a,
                 permute_listener_v=False, permute_listener_a=False,
                 rng=None):
        self.path_list_v = [str(p) for p in path_list_v]
        self.path_list_a = [str(p) for p in path_list_a]
        self.feat_v = feat_v
        self.feat_a = feat_a
        self.target_size_v = target_size_v
        self.target_size_a = target_size_a

        self.files  = list(sorted(file_list))
        self.labels = [labels_json[Path(f).stem] for f in self.files]

        # Per-slot file lists. Primary = driven by sample identity (carries
        # the label). Listener file list is separate only when length-2;
        # for length-1 slots, the primary IS the listener if the path is
        # a listener path.
        self.files_v_primary  = list(self.files)
        self.files_v_listener = list(self.files)
        self.files_a_primary  = list(self.files)
        self.files_a_listener = list(self.files)

        rng = rng or np.random.default_rng(42)
        if permute_listener_v or permute_listener_a:
            # Shared permutation across modalities so the fake listener
            # is coherent in face + voice (row 9-shuffled convention).
            sigma = np.arange(len(self.files))
            rng.shuffle(sigma)
            permuted = [self.files[s] for s in sigma]

            def apply(primary, listener, is_concat):
                if is_concat:
                    return primary, permuted
                return permuted, listener

            if permute_listener_v:
                self.files_v_primary, self.files_v_listener = apply(
                    self.files_v_primary, self.files_v_listener,
                    is_concat=(len(path_list_v) == 2))
            if permute_listener_a:
                self.files_a_primary, self.files_a_listener = apply(
                    self.files_a_primary, self.files_a_listener,
                    is_concat=(len(path_list_a) == 2))

    @staticmethod
    def _pad(x, feat_dim, target_size):
        T = x.shape[0]
        if T > target_size:
            x = x[:target_size]
        elif T < target_size:
            x = np.vstack([x, np.zeros((target_size - T, feat_dim), np.float32)])
        return x

    def _load_slot(self, path_list, fname_primary, fname_listener,
                   feat_dim, target_size):
        if len(path_list) == 1:
            x = np.load(os.path.join(path_list[0], fname_primary)).astype(np.float32)
            return self._pad(x, feat_dim, target_size)
        per = feat_dim // 2
        xp = np.load(os.path.join(path_list[0], fname_primary)).astype(np.float32)
        xs = np.load(os.path.join(path_list[1], fname_listener)).astype(np.float32)
        xp = self._pad(xp, per, target_size)
        xs = self._pad(xs, per, target_size)
        return np.concatenate([xp, xs], axis=-1)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        x_v = self._load_slot(
            self.path_list_v,
            self.files_v_primary[idx], self.files_v_listener[idx],
            self.feat_v, self.target_size_v)
        x_a = self._load_slot(
            self.path_list_a,
            self.files_a_primary[idx], self.files_a_listener[idx],
            self.feat_a, self.target_size_a)
        return torch.tensor(x_v), torch.tensor(x_a), self.labels[idx]


# ── MergedDataset: early-fusion loader (feature-concat at aligned rate) ─
# OpenFace runs at 30 fps (MAX_LEN_V = 60 per 2 s window), WavLM at 10 Hz
# (MAX_LEN_A = 20). For early fusion we need a COMMON timestep — the
# cheapest alignment is to mean-pool OpenFace ×3 to 10 Hz (spec §6.2.c).
# Result shape: (MAX_LEN_EARLY = 20, feat_v + feat_a).
class MergedDataset(Dataset):
    """Early-fusion dataset. OpenFace is mean-pooled ×3 to match WavLM's
    10 Hz rate, then concatenated on the feature axis."""
    def __init__(self, path_list_v, path_list_a, file_list, labels_json,
                 feat_v, feat_a, target_size_early,
                 permute_listener_v=False, permute_listener_a=False,
                 rng=None):
        # Re-use BimodalDataset machinery; we'll pool visual in __getitem__.
        self.bm = BimodalDataset(
            path_list_v, path_list_a, file_list, labels_json,
            feat_v, feat_a,
            target_size_v=MAX_LEN_V, target_size_a=MAX_LEN_A,
            permute_listener_v=permute_listener_v,
            permute_listener_a=permute_listener_a,
            rng=rng,
        )
        self.target_size_early = target_size_early

    def __len__(self):
        return len(self.bm)

    def __getitem__(self, idx):
        x_v, x_a, label = self.bm[idx]                     # (60, feat_v), (20, feat_a)
        # Mean-pool OpenFace ×3: (60, feat_v) → (20, 3, feat_v) → (20, feat_v).
        # Use .reshape (not .view) to handle non-contiguous tensors safely
        x_v_pooled = x_v.reshape(self.target_size_early, 3, -1).mean(dim=1)
        x = torch.cat([x_v_pooled, x_a], dim=-1)           # (20, feat_v + feat_a)
        return x, label


> **Sanity check ahead.** Following cell instantiates all four loader types for **row 1** (the primary dyadic cross-pair experiment) and prints a batch shape from each. Before training, eyeball that the shapes match the constants block — if OpenFace comes out `(32, 60, 20)` and WavLM `(32, 20, 768)`, the feature-building script and these loaders agree on rate and dimensionality.

In [ ]:
# ── Loader factory — parameterized on an EXPERIMENTS config dict ───────
def _append_split(path_list, split):
    """Append /{split} to each root path. Every model_input feature dir
    has {train, val, test} subdirs; the EXPERIMENTS config stores the
    roots so every builder has to do this last join."""
    return [os.path.join(str(p), split) for p in path_list]


def _permute_flags(exp):
    """Translate exp['permute_v_at_test'] into (permute_v, permute_a).
    Listener is always in the visual slot (when visual present); listener
    is ALSO in the audio slot when audio is a length-2 concat (row 9-shuffled).
    """
    if not exp.get('permute_v_at_test', False):
        return False, False
    pv = exp.get('feat_path_v') is not None
    pa = (exp.get('feat_path_a') is not None
          and len(exp['feat_path_a']) == 2)
    return pv, pa


def _seq_loader(path_list, file_list, labels_json, feat_total, target_size,
                shuffle, permute=False, rng=None):
    ds = SequenceDataset(path_list, file_list, labels_json,
                         feat_total, target_size,
                         permute_listener=permute, rng=rng)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=(device.type == 'cuda'))


def _bi_loader(plv, pla, file_list, labels_json, fv, fa, tv, ta,
               shuffle, permute_v=False, permute_a=False, rng=None):
    ds = BimodalDataset(plv, pla, file_list, labels_json,
                        fv, fa, tv, ta,
                        permute_listener_v=permute_v,
                        permute_listener_a=permute_a, rng=rng)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=(device.type == 'cuda'))


def _merged_loader(plv, pla, file_list, labels_json, fv, fa, te,
                   shuffle, permute_v=False, permute_a=False, rng=None):
    ds = MergedDataset(plv, pla, file_list, labels_json, fv, fa, te,
                       permute_listener_v=permute_v,
                       permute_listener_a=permute_a, rng=rng)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=(device.type == 'cuda'))


def make_loaders_for_experiment(exp, splits_dict, labels_json,
                                test_only_permute=True):
    """Returns {loader_kind: (train, val, test)}.

    Only builds loaders this experiment's models actually need. When
    test_only_permute=True (default), shuffle flags are applied ONLY to
    the test loader — train and val see the real dyadic pairing so the
    model learns the true distribution (spec §8.2).
    """
    # late_fusion blends softmax probs from unimodal_v + unimodal_a, so
    # whenever late_fusion is requested we also need BOTH unimodal loaders
    # — even if the experiment doesn't explicitly train those models
    # (rows 6 and 9-shuffled reuse source unimodals).
    needs_lf = 'late_fusion' in exp['models']
    need_uni_v = ('unimodal_v' in exp['models']) or needs_lf
    need_uni_a = ('unimodal_a' in exp['models']) or needs_lf
    need_bim = any(m in exp['models'] for m in ('neural_concat', 'late_fusion'))
    need_merged = 'early_fusion' in exp['models']
    pv, pa = _permute_flags(exp)
    out = {}

    def triple(shuffle_train, builder):
        tr = builder('train', shuffle=shuffle_train, perm=False)
        va = builder('val', shuffle=False, perm=False)
        te = builder('test', shuffle=False, perm=(pv or pa))
        return tr, va, te

    if need_uni_v and exp.get('feat_path_v') is not None:
        def b(split, shuffle, perm):
            return _seq_loader(
                _append_split(exp['feat_path_v'], split),
                splits_dict[split], labels_json,
                feat_total=exp['feat_v'], target_size=MAX_LEN_V,
                shuffle=shuffle, permute=(perm and pv))
        out['unimodal_v'] = triple(True, b)

    if need_uni_a and exp.get('feat_path_a') is not None:
        def b(split, shuffle, perm):
            return _seq_loader(
                _append_split(exp['feat_path_a'], split),
                splits_dict[split], labels_json,
                feat_total=exp['feat_a'], target_size=MAX_LEN_A,
                shuffle=shuffle, permute=(perm and pa))
        out['unimodal_a'] = triple(True, b)

    if need_bim:
        def b(split, shuffle, perm):
            return _bi_loader(
                _append_split(exp['feat_path_v'], split),
                _append_split(exp['feat_path_a'], split),
                splits_dict[split], labels_json,
                fv=exp['feat_v'], fa=exp['feat_a'],
                tv=MAX_LEN_V, ta=MAX_LEN_A,
                shuffle=shuffle,
                permute_v=(perm and pv), permute_a=(perm and pa))
        out['bimodal'] = triple(True, b)

    if need_merged:
        def b(split, shuffle, perm):
            return _merged_loader(
                _append_split(exp['feat_path_v'], split),
                _append_split(exp['feat_path_a'], split),
                splits_dict[split], labels_json,
                fv=exp['feat_v'], fa=exp['feat_a'],
                te=MAX_LEN_EARLY,
                shuffle=shuffle,
                permute_v=(perm and pv), permute_a=(perm and pa))
        out['merged'] = triple(True, b)

    return out


# ── Sanity check on row 1 ──────────────────────────────────────────────
_exp1 = EXPERIMENTS['row1_dyadic_cross_pair']
_loaders1 = make_loaders_for_experiment(_exp1, splits, TRAIN_LABELS)
print(f"Row 1 loaders built: {list(_loaders1.keys())}")
for kind, (tr, _, _) in _loaders1.items():
    batch = next(iter(tr))
    if kind == 'bimodal':
        xvb, xab, yb = batch
        print(f"  {kind:11s} train batch Xv={tuple(xvb.shape)} "
              f"Xa={tuple(xab.shape)} y={tuple(yb.shape)}")
    else:
        xb, yb = batch
        print(f"  {kind:11s} train batch X={tuple(xb.shape)} y={tuple(yb.shape)}")


---
## Section 3: Unimodal Baselines

### 3.1 Training and evaluation helpers

In PyTorch there is no `.fit `. We write the training loop ourselves. 
This gives you full control but requires a few boilerplate steps every epoch:

```
for each batch:
 optimizer.zero_grad # clear gradients from previous step
 logits = model(X) # forward pass
 loss = criterion(logits, y)
 loss.backward # compute gradients
 optimizer.step # update weights
```

Key rules:
- Call `model.train ` before the training loop and `model.eval ` before validation
- Wrap validation/inference in `torch.no_grad ` to skip gradient bookkeeping
- `nn.CrossEntropyLoss` expects **raw logits** (no softmax), not probabilities

In [ ]:
# ── Loss function ──────────────────────────────────────────────────────
# Class-weighted cross-entropy over raw logits. Weights are inverse-
# frequency on the train split so rare BC (~15%) gets a larger gradient
from collections import Counter as _Counter
_train_class_counts = _Counter(TRAIN_LABELS[f[:-4]] for f in splits['train'])
_total = float(sum(_train_class_counts.values()))
_class_weights = torch.tensor(
    [_total / (NUM_CLASSES * max(_train_class_counts.get(i, 1), 1))
     for i in range(NUM_CLASSES)],
    dtype=torch.float32,
    device=device,
)
print(f'Class weights (inverse-frequency on train): {_class_weights.cpu().numpy()}')
criterion = nn.CrossEntropyLoss(weight=_class_weights)


def run_epoch(model, loader, optimizer=None, bimodal=False, return_preds=False):
    """One training (if optimizer given) or eval (if None) epoch.
    return_preds=True also returns (preds, labels) numpy arrays so the
    caller can compute additional metrics (e.g. macro-F1) without a
    second pass through the loader."""
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_preds = [] if return_preds else None
    all_labels = [] if return_preds else None
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            if bimodal:
                x_v, x_a, y = batch
                x_v, x_a, y = x_v.to(device), x_a.to(device), y.to(device)
                logits = model(x_v, x_a)
            else:
                x, y = batch
                x, y = x.to(device), y.to(device)
                logits = model(x)
            loss = criterion(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y)
            pred = logits.argmax(1)
            correct    += (pred == y).sum().item()
            n          += len(y)
            if return_preds:
                all_preds.append(pred.cpu().numpy())
                all_labels.append(y.cpu().numpy())
    if return_preds:
        return (total_loss / n, correct / n,
                (np.concatenate(all_preds), np.concatenate(all_labels)))
    return total_loss / n, correct / n


def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR,
                patience=4, bimodal=False):
    """Full training loop with early stopping on VAL MACRO-F1.
    Macro-F1 is the reported metric and is a better signal than val
    accuracy on our imbalanced 3-class setup (natural rate + class-weighted loss).
"""
    optimizer  = torch.optim.Adam(model.parameters(), lr=lr)
    best_f1    = -1.0
    best_epoch = -1
    best_state = None
    no_improve = 0
    history = {'train_loss':   [], 'train_acc':    [],
               'val_loss':     [], 'val_acc':      [],
               'val_macro_f1': []}
    for epoch in range(epochs):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, bimodal)
        vl_loss, vl_acc, (vp, vl) = run_epoch(
            model, val_loader, None, bimodal, return_preds=True)
        vl_macro_f1 = float(f1_score(
            vl, vp, labels=[0, 1, 2], average='macro', zero_division=0))
        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss);   history['val_acc'].append(vl_acc)
        history['val_macro_f1'].append(vl_macro_f1)
        print(f'  Epoch {epoch+1:02d}/{epochs}  '
              f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  '
              f'val_loss={vl_loss:.4f}  val_acc={vl_acc:.3f}  '
              f'val_macroF1={vl_macro_f1:.3f}')
        if vl_macro_f1 > best_f1:
            best_f1, best_epoch = vl_macro_f1, epoch
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping at epoch {epoch+1} '
                      f'(best at epoch {best_epoch+1}, macroF1={best_f1:.3f})')
                break
    model.load_state_dict(best_state)
    return history


@torch.no_grad()
def evaluate(model, loader, name, bimodal=False, class_ids=(0, 1, 2)):
    """Collect predictions; compute accuracy + per-class F1 + macro-F1.
    Stores {'acc', 'macro_f1', 'f1_per_class': [H,Y,B]} in results_table[name]."""
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        if bimodal:
            x_v, x_a, y = batch
            logits = model(x_v.to(device), x_a.to(device))
        else:
            x, y = batch
            logits = model(x.to(device))
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_labels.append(y.numpy())
    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    acc    = accuracy_score(labels, preds) * 100
    f1_per = f1_score(labels, preds, labels=list(class_ids),
                      average=None, zero_division=0) * 100
    macro_f1 = float(f1_per.mean())
    results_table[name] = {
        'acc':          float(acc),
        'macro_f1':     macro_f1,
        'f1_per_class': f1_per.tolist(),
    }
    print(f'[{name}] acc={acc:5.2f}%  macro-F1={macro_f1:5.2f}  '
          f'F1(HOLD/YIELD/BC)=['
          f'{f1_per[0]:.1f}, {f1_per[1]:.1f}, {f1_per[2]:.1f}]')
    return preds, labels


### 3.2 Exercise — Build the GRU Classifier

Complete `GRUClassifier` below. The target architecture:

```
Input (batch, MAX_LEN, feature_dim)
 → nn.GRU(feature_dim, hidden_size, batch_first=True)
 → take last hidden state h_n[-1] → shape (batch, hidden_size)
 → nn.Linear(hidden_size, hidden_size) + ReLU
 → nn.Dropout(0.3)
 → nn.Linear(hidden_size, num_classes) ← raw logits, NO softmax
```

> **Why no softmax here?** `nn.CrossEntropyLoss` already applies `log_softmax` internally. 
> At inference, use `logits.argmax(dim=1)` for the predicted class.

### Unimodal GRU

In [ ]:
class GRUClassifier(nn.Module):
    """Unimodal GRU sequence classifier. Unchanged from practicum.

    Args:
        input_size:  feature dimension per timestep
        hidden_size: number of GRU hidden units
        num_classes: number of output classes
    """
    def __init__(self, input_size, hidden_size=GRU_UNITS, num_classes=NUM_CLASSES):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, h_n = self.gru(x)      # h_n: (1, batch, hidden)
        h = h_n[-1]               # (batch, hidden)
        return self.fc2(self.drop(self.relu(self.fc1(h))))


# Smoke tests for all (feat_dim, max_len) pairs we'll actually use:
#   (20,   60) → row-1 visual (listener face)
#   (40,   60) → row-9 visual (concat, speaker+listener face)
#   (768,  20) → row-1 audio  (speaker audio)
#   (1536, 20) → row-9 audio  (concat, speaker+listener audio)
for feat_dim, max_len in [(20, MAX_LEN_V), (40, MAX_LEN_V),
                          (768, MAX_LEN_A), (1536, MAX_LEN_A)]:
    m = GRUClassifier(input_size=feat_dim).to(device)
    dummy = torch.zeros(4, max_len, feat_dim).to(device)
    out = m(dummy)
    assert out.shape == (4, NUM_CLASSES), f"feat_dim={feat_dim}: {out.shape}"
    print(f"  Shape OK  input_size={feat_dim:4d} max_len={max_len:2d} "
          f"→ logits {tuple(out.shape)}")


In [ ]:
# ── Row 1 — Unimodal visual (LISTENER's face on listener stream) ──────
# Loaders come from Cell 12 (_loaders1['unimodal_v']). Variable names
# (model_AU, train_of, ...) follow the practicum's convention so the
# late-fusion code later reuses them unchanged.
train_of, val_of, test_of = _loaders1['unimodal_v']

print('=== Row 1 — Unimodal visual (listener face, OpenFace) ===')
model_AU = GRUClassifier(input_size=_exp1['feat_v']).to(device)
history_AU = train_model(model_AU, train_of, val_of)

In [ ]:
# ── Row 1 — Unimodal audio (SPEAKER's audio on speaker stream) ────────
train_wavlm, val_wavlm, test_wavlm = _loaders1['unimodal_a']

print('=== Row 1 — Unimodal audio (speaker audio, WavLM-base+) ===')
model_wavlm = GRUClassifier(input_size=_exp1['feat_a']).to(device)
history_wavlm = train_model(model_wavlm, train_wavlm, val_wavlm)


In [ ]:
# Evaluate on the row 1 test set. evaluate() now stores a rich dict:
#   results_table[name] = {'acc', 'macro_f1', 'f1_per_class': [H,Y,B]}
preds_AU,    labels_AU    = evaluate(model_AU,    test_of,    'row1_unimodal_v')
preds_wavlm, labels_wavlm = evaluate(model_wavlm, test_wavlm, 'row1_unimodal_a')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for hist, name, ax in zip(
    [history_AU, history_wavlm],
    ['Listener face (OpenFace, 20-dim)',
     'Speaker audio (WavLM-base+, 768-dim)'],
    axes,
):
    ax.plot(hist['train_acc'], label='train')
    ax.plot(hist['val_acc'],   label='val')
    ax.set_title(f'{name}\ntraining accuracy')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


> **What to look for.** Training accuracy rises with class-weighted cross-entropy pushing gradient toward YIELD and BC (inverse-frequency weights from Cell 14). Val and test both run at the natural (HOLD-dominant) distribution, so val accuracy will plateau near the HOLD base rate — **macro-F1** (reported above) is the honest signal, not accuracy. Expect the audio stream to beat the visual stream on this task (audio carries prosodic turn-end cues the 20-dim OpenFace summary does not); the interesting comparison is what the **fusion models** (next sections) do with them together, and how per-class F1 degrades across the **τ-sweep** (added at the end).

### Bimodal GRU

---
## Section 4: Early Fusion (row 1)

**Idea.** Concatenate all modality features at the **input** before any learned layer. A single model then learns from the full `(T, FEAT_V + FEAT_A)` sequence.
OpenFace (60, 20) ── mean-pool ×3 → (20, 20) ──┐
 concat → (20, 788) → GRU → Dense → logits
WavLM (20, 768) ────────────────── (20, 768) ──┘


OpenFace is mean-pooled ×3 to match WavLM's 10 Hz rate before feature-axis concatenation (spec §6.2.c). `MAX_LEN_EARLY = 20`. Row 9 (later) will use the same structure but with `(20, 40+1536) = (20, 1576)`.

In [ ]:
class EarlyFusionGRU(nn.Module):
    """Early fusion GRU: input = concat(visual, audio) on feature axis,
    after rate alignment. input_size must be explicitly passed (row 1: 788,
    row 9: 1576)."""
    def __init__(self, input_size, hidden_size=128, num_classes=NUM_CLASSES):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=3,
                          batch_first=True, dropout=0.3)
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, h_n = self.gru(x)
        return self.fc2(self.drop(self.relu(self.fc1(h_n[-1]))))


# ── Row 1 — Early fusion training ────────────────────────────────────
train_ef, val_ef, test_ef = _loaders1['merged']

print('=== Row 1 — Early Fusion ===')
model_ef = EarlyFusionGRU(input_size=_exp1['feat_v'] + _exp1['feat_a']).to(device)
history_ef = train_model(model_ef, train_ef, val_ef)
preds_ef, labels_ef = evaluate(model_ef, test_ef, 'row1_early_fusion')


> **Early-fusion takeaway.** The GRU sees both modalities jointly at every timestep; gradient flow can couple them. Expected: macro-F1 ≥ unimodal audio alone (if visual adds signal), but not dramatically — our 20-dim OpenFace is low-capacity compared to 768-dim WavLM and gets dominated at the input. Late fusion (next) and Neural-Concat (after) give each modality its own encoder first, which can help the weaker modality matter more.

---
## Section 5: Late Fusion

**Idea:** Train separate unimodal models (already done in Section 3), then combine their **softmax probability vectors** at inference time. No new model is trained for the fusion itself.

```
OpenFace → GRU → softmax ──┐
 weighted sum → argmax → prediction
WavLM → GRU → softmax ──┘
```


### Exercise 5.1 — Uniform average (w = 0.5)

In [ ]:
@torch.no_grad()
def get_probs(model, loader, bimodal=False):
    """Return softmax probabilities + labels over an entire DataLoader."""
    model.eval()
    all_probs, all_labels = [], []
    for batch in loader:
        if bimodal:
            x_v, x_a, y = batch
            logits = model(x_v.to(device), x_a.to(device))
        else:
            x, y = batch
            logits = model(x.to(device))
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        all_labels.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


def late_fusion_predict(probs_v, probs_a, w=0.5):
    """Combine predictions by weighted average of softmax probabilities.
    w = weight for visual, (1-w) = weight for audio."""
    combined = w * probs_v + (1 - w) * probs_a
    return combined.argmax(axis=1)


def _store_late_fusion(name, preds, labels):
    """Mirror evaluate()'s metric schema for late-fusion results."""
    acc    = accuracy_score(labels, preds) * 100
    f1_per = f1_score(labels, preds, labels=[0, 1, 2],
                      average=None, zero_division=0) * 100
    macro_f1 = float(f1_per.mean())
    results_table[name] = {
        'acc':          float(acc),
        'macro_f1':     macro_f1,
        'f1_per_class': f1_per.tolist(),
    }
    print(f'[{name}] acc={acc:5.2f}%  macro-F1={macro_f1:5.2f}  '
          f'F1(HOLD/YIELD/BC)=[{f1_per[0]:.1f}, {f1_per[1]:.1f}, {f1_per[2]:.1f}]')


# Pre-compute probs once for row 1 (reused in the grid search in Cell 29).
val_probs_AU,    val_labels  = get_probs(model_AU,    val_of)
val_probs_wavlm, _           = get_probs(model_wavlm, val_wavlm)
test_probs_AU,   test_labels = get_probs(model_AU,    test_of)
test_probs_wavlm, _          = get_probs(model_wavlm, test_wavlm)

# Uniform average (w = 0.5).
lf_preds_unif = late_fusion_predict(test_probs_AU, test_probs_wavlm, w=0.5)
_store_late_fusion('row1_late_fusion_avg', lf_preds_unif, test_labels)


### Exercise 5.2 — Learned blend weight

Sweep `w ∈ [0, 1]` on the **validation** set and pick the weight that maximises val accuracy. Apply that weight on the **test** set.

In [ ]:
# ── Grid search over w on validation set — ON MACRO-F1, not accuracy. ──
# our imbalanced 3-class setup.
best_w, best_val_f1 = 0.5, -1.0
ws, val_f1s = [], []
for w in np.linspace(0, 1, 21):
    preds = late_fusion_predict(val_probs_AU, val_probs_wavlm, w=w)
    f1    = f1_score(val_labels, preds, labels=[0, 1, 2],
                     average='macro', zero_division=0)
    ws.append(w); val_f1s.append(f1 * 100)
    if f1 > best_val_f1:
        best_val_f1, best_w = f1, w
print(f'Best w (visual weight): {best_w:.2f}  val macro-F1={best_val_f1*100:.2f}%')

# Apply best w to test.
lf_preds_w = late_fusion_predict(test_probs_AU, test_probs_wavlm, w=best_w)
_store_late_fusion(f'row1_late_fusion_w={best_w:.2f}', lf_preds_w, test_labels)

plt.figure(figsize=(7, 4))
plt.plot(ws, val_f1s, marker='o')
plt.axvline(best_w, color='red', ls='--', label=f'best w={best_w:.2f}')
plt.xlabel('w  (weight on visual / listener face)')
plt.ylabel('Val macro-F1 (%)')
plt.title('Row 1 Late Fusion: weight sweep on validation set')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Note on cell ordering: `best_w` (global) is consumed by Cell 33's
# preds_map key and by Cell 35's trained_models registration. Re-run
# Cells 27 → 29 sequentially; do NOT re-run Cell 35 alone without


---
## Section 6: Neural Concatenation Fusion (row 1, intermediate fusion)

**Idea.** Give each modality its own GRU encoder. After the GRU extracts a compact representation from each, **concatenate the hidden states** and pass them through shared FC layers. The whole model is trained jointly end-to-end — neither modality's features dominate the input vector (unlike early fusion where WavLM's 768 dims swamp OpenFace's 20).

OpenFace (B, 60, 20) → GRU → h_v (B, H) ──┐
 
 Cat(B, 2H) → FC → logits

WavLM (B, 20, 768) → GRU → h_a (B, H) ──┘

In [ ]:
class NeuralConcatFusion(nn.Module):
    """Dual-branch GRU with hidden-state concatenation. Unchanged from
    practicum — feat_v / feat_a are constructor args so this adapts to
    any experiment's dimensionality."""
    def __init__(self, feat_v, feat_a,
                 hidden_size=GRU_UNITS, num_classes=NUM_CLASSES):
        super().__init__()
        self.gru_v = nn.GRU(feat_v, hidden_size, batch_first=True)
        self.gru_a = nn.GRU(feat_a, hidden_size, batch_first=True)
        self.fc1 = nn.Linear(hidden_size * 2, hidden_size)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x_v, x_a):
        _, h_v = self.gru_v(x_v); h_v = h_v[-1]
        _, h_a = self.gru_a(x_a); h_a = h_a[-1]
        return self.fc2(self.drop(self.relu(self.fc1(torch.cat([h_v, h_a], dim=-1)))))


# ── Row 1 — Neural Concat Fusion ─────────────────────────────────────
train_bm, val_bm, test_bm = _loaders1['bimodal']

print('=== Row 1 — Neural Concatenation Fusion ===')
model_nc = NeuralConcatFusion(feat_v=_exp1['feat_v'],
                              feat_a=_exp1['feat_a']).to(device)
history_nc = train_model(model_nc, train_bm, val_bm, bimodal=True)
preds_nc, labels_nc = evaluate(model_nc, test_bm, 'row1_neural_concat', bimodal=True)


---
## Section 7: Row 1 Results

Aggregates row-1 results across the five models trained or assembled above (unimodal visual, unimodal audio, early fusion, late fusion avg, late fusion best-w, neural concat). Later sections will run rows 3, 9, 6, 9-shuffled and the horizon (τ-sweep) curve.

In [ ]:
row1_methods = {k: v for k, v in results_table.items() if k.startswith('row1_')}

# Sort by macro-F1 descending for display order.
row1_sorted = sorted(row1_methods.items(),
                     key=lambda kv: kv[1]['macro_f1'], reverse=True)

print('\n' + '=' * 72)
print(f'{"Method":<36} {"Acc":>7} {"MacroF1":>8}   F1 HOLD / YIELD / BC')
print('=' * 72)
for name, m in row1_sorted:
    per = m['f1_per_class']
    print(f'{name:<36} {m["acc"]:>6.2f}% {m["macro_f1"]:>7.2f}   '
          f'{per[0]:>5.1f} / {per[1]:>5.1f} / {per[2]:>5.1f}')
print('=' * 72)

# Bar chart — macro-F1 by method.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names = [n for n, _ in row1_sorted]
macros = [m['macro_f1'] for _, m in row1_sorted]
bars = axes[0].bar(names, macros, edgecolor='black', linewidth=0.5)
axes[0].set_ylim(0, 100)
for bar, val in zip(bars, macros):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f'{val:.1f}', ha='center', fontsize=9)
axes[0].set_ylabel('Macro-F1')
axes[0].set_title('Row 1 Fusion Comparison (macro-F1 on test)')
axes[0].tick_params(axis='x', rotation=30)

# Confusion matrix of the best method.
best_name, best_metrics = row1_sorted[0]
preds_map = {
    'row1_unimodal_v':          (preds_AU,       labels_AU),
    'row1_unimodal_a':          (preds_wavlm,    labels_wavlm),
    'row1_early_fusion':        (preds_ef,       labels_ef),
    'row1_neural_concat':       (preds_nc,       labels_nc),
    'row1_late_fusion_avg':     (lf_preds_unif,  test_labels),
    f'row1_late_fusion_w={best_w:.2f}': (lf_preds_w, test_labels),
}
if best_name in preds_map:
    p, l = preds_map[best_name]
    cm = confusion_matrix(l, p, labels=[0, 1, 2])
    ConfusionMatrixDisplay(cm, display_labels=label_names).plot(
        ax=axes[1], cmap='Blues', colorbar=False)
    axes[1].set_title(f'Confusion Matrix — {best_name}')
    axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout(); plt.show()


---
## Section 8: Remaining Experiments (rows 3, 6, 9, 9-shuffled)

Row 1 is done. We now run:

- **Row 3** — unimodal-audio baseline (trains one `GRUClassifier` on speaker WavLM features, no visual). This is the audio-only floor that row 1's fusion models should improve on.
- **Row 9** — MM-VAP-style full dyadic (concat speaker+listener in both visual and audio). Same 5 models as row 1 but at `FEAT_V=40, FEAT_A=1536`.
- **Row 6** — permuted-dyad control on row 1. **No training** — reuses row 1's trained models and evaluates them on a test loader with listener-face filenames shuffled. Spec §8.2: if performance drops vs. row 1, the dyadic-coordination claim is supported.
- **Row 9-shuffled** — permuted-dyad control on row 9. Same reuse-and-permute pattern. Tests whether the listener half of the concat'd features was actually informative.

In [ ]:
# ── Persistent trained-model registry across all experiments ──────────
# Row 1's models are trained in Cells 18-31; register them here plus
# Cell 29's best_w so row 6 can reuse the SAME w (spec §8.2 —
# "same trained model" includes the late-fusion hyperparameter).
trained_models = {
    'row1_dyadic_cross_pair': {
        'unimodal_v':      model_AU,
        'unimodal_a':      model_wavlm,
        'early_fusion':    model_ef,
        'neural_concat':   model_nc,
        '_late_fusion_w':  best_w,           # set by Cell 29
    }
}


def _train_one(kind, exp, loaders):
    """Train a single model for one (exp, kind) combination.
    Returns the trained model."""
    if kind == 'unimodal_v':
        m = GRUClassifier(input_size=exp['feat_v']).to(device)
        tr, va, _ = loaders['unimodal_v']
        train_model(m, tr, va)
        return m
    if kind == 'unimodal_a':
        m = GRUClassifier(input_size=exp['feat_a']).to(device)
        tr, va, _ = loaders['unimodal_a']
        train_model(m, tr, va)
        return m
    if kind == 'early_fusion':
        m = EarlyFusionGRU(input_size=exp['feat_v'] + exp['feat_a']).to(device)
        tr, va, _ = loaders['merged']
        train_model(m, tr, va)
        return m
    if kind == 'neural_concat':
        m = NeuralConcatFusion(feat_v=exp['feat_v'], feat_a=exp['feat_a']).to(device)
        tr, va, _ = loaders['bimodal']
        train_model(m, tr, va, bimodal=True)
        return m
    raise ValueError(f'Unknown model kind: {kind}')


def _eval_one(kind, exp_name, model, loaders):
    """Evaluate one trained model on the experiment's test loader.
    Returns (preds, labels). Updates results_table[{exp_name}_{kind}]."""
    if kind == 'unimodal_v':
        _, _, te = loaders['unimodal_v']
        return evaluate(model, te, f'{exp_name}_{kind}')
    if kind == 'unimodal_a':
        _, _, te = loaders['unimodal_a']
        return evaluate(model, te, f'{exp_name}_{kind}')
    if kind == 'early_fusion':
        _, _, te = loaders['merged']
        return evaluate(model, te, f'{exp_name}_{kind}')
    if kind == 'neural_concat':
        _, _, te = loaders['bimodal']
        return evaluate(model, te, f'{exp_name}_{kind}', bimodal=True)
    raise ValueError(f'Unknown model kind: {kind}')


def _eval_late_fusion(exp_name, m_v, m_a, loaders, preselected_w=None):
    """Late-fusion eval.

    If preselected_w is None (train mode, rows 1/3/9), grid-searches w on
    val macro-F1 and applies best-w to test.

    If preselected_w is a float (reuse mode, rows 6/9-shuffled), SKIPS
    grid search and applies that w directly. This preserves "same trained
    model" semantics for the permuted-dyad control.
    """
    _, vu_v, tu_v = loaders['unimodal_v']
    _, vu_a, tu_a = loaders['unimodal_a']
    tpv, tlab = get_probs(m_v, tu_v)
    tpa, _    = get_probs(m_a, tu_a)

    if preselected_w is not None:
        best_w_local = preselected_w
        suffix = f'{best_w_local:.2f}_reused'
    else:
        vpv, vlab = get_probs(m_v, vu_v)
        vpa, _    = get_probs(m_a, vu_a)
        best_w_local = 0.5
        best_f1 = -1.0
        for w in np.linspace(0, 1, 21):
            p = late_fusion_predict(vpv, vpa, w=w)
            f1 = f1_score(vlab, p, labels=[0, 1, 2],
                          average='macro', zero_division=0)
            if f1 > best_f1:
                best_f1, best_w_local = f1, w
        suffix = f'{best_w_local:.2f}'

    preds_w = late_fusion_predict(tpv, tpa, w=best_w_local)
    _store_late_fusion(f'{exp_name}_late_fusion_w={suffix}', preds_w, tlab)
    return best_w_local, preds_w, tlab


def run_experiment(exp_name):
    """End-to-end: build loaders, train (or reuse) models, evaluate.
    Populates trained_models[exp_name] and results_table."""
    exp = EXPERIMENTS[exp_name]
    print(f'\n{"=" * 72}\nExperiment: {exp_name}\n  {exp["description"]}\n{"=" * 72}')

    loaders = make_loaders_for_experiment(exp, splits, TRAIN_LABELS)

    # Weight-reuse path (rows 6 and 9-shuffled).
    if exp['reuse_weights_from']:
        src = exp['reuse_weights_from']
        source_models = trained_models[src]
        print(f'Reusing trained weights from "{src}"; eval-only on this test loader.')
        for kind in exp['models']:
            if kind == 'late_fusion':
                continue
            if kind not in source_models:
                print(f'  WARN: source "{src}" did not train {kind}; skipping.')
                continue
            _eval_one(kind, exp_name, source_models[kind], loaders)
        # Late fusion with SOURCE's best_w (no re-tuning) — spec §8.2.
        if 'late_fusion' in exp['models'] \
                and 'unimodal_v' in source_models \
                and 'unimodal_a' in source_models:
            src_w = source_models.get('_late_fusion_w', None)
            _eval_late_fusion(exp_name, source_models['unimodal_v'],
                              source_models['unimodal_a'], loaders,
                              preselected_w=src_w)
        # Shallow copy so mutations on trained_models[exp_name] don't
        trained_models[exp_name] = dict(source_models)
        return loaders

    # Training path (rows 3 and 9).
    models = {}
    for kind in exp['models']:
        if kind == 'late_fusion':
            continue
        print(f'\n--- Training {exp_name} / {kind} ---')
        models[kind] = _train_one(kind, exp, loaders)
        _eval_one(kind, exp_name, models[kind], loaders)
    if 'late_fusion' in exp['models']:
        print(f'\n--- Late fusion {exp_name} ---')
        best_w_this, _, _ = _eval_late_fusion(
            exp_name, models['unimodal_v'], models['unimodal_a'], loaders)
        models['_late_fusion_w'] = best_w_this
    trained_models[exp_name] = models
    return loaders


In [ ]:
loaders_row3 = run_experiment('row3_unimodal_audio')

In [ ]:
loaders_row9 = run_experiment('row9_mmvap_full_dyadic')

In [ ]:
loaders_row6 = run_experiment('row6_permuted_dyad_control')

In [ ]:
loaders_row9s = run_experiment('row9_shuffled_control')

---
## Section 9: Horizon Curve — τ-sweep evaluation

All trained models (from rows 1, 3, 9) are re-evaluated on test at each τ ∈ {100, 200, 400, 500, 800, 1600} ms. Input features are τ-invariant (spec §11.1) — only labels change per τ. Per-class F1 vs. τ is the main scientific output: how far in advance can the multimodal stream predict turn-taking events?

Rows 6 and 9-shuffled reuse their parent's trained weights, so their τ-curves show the degradation effect of breaking the dyadic pairing at each lead time.

In [ ]:
tau_curves = {}  # {exp_name: {kind: {tau_ms: metrics_dict}}}


def _test_loader_at_tau(exp, kind, tau_ms, labels_at_tau):
    """Build a test loader for one (exp, kind, τ) combination.

    File list is enumerated FROM DISK (not splits['test'], which was
    pre-filtered at τ=400) and filtered to basenames whose τ-label is
    in {0, 1, 2}. Features are unchanged across τ.
    """
    all_test = sorted(f for f in os.listdir(ENUM_DIR / 'test') if f.endswith('.npy'))
    test_files = [f for f in all_test
                  if labels_at_tau.get(f[:-4], 999) in (0, 1, 2)]
    pv, pa = _permute_flags(exp)
    plv = exp.get('feat_path_v')
    pla = exp.get('feat_path_a')

    if kind == 'unimodal_v':
        return _seq_loader(_append_split(plv, 'test'), test_files, labels_at_tau,
                           feat_total=exp['feat_v'], target_size=MAX_LEN_V,
                           shuffle=False, permute=pv)
    if kind == 'unimodal_a':
        return _seq_loader(_append_split(pla, 'test'), test_files, labels_at_tau,
                           feat_total=exp['feat_a'], target_size=MAX_LEN_A,
                           shuffle=False, permute=pa)
    if kind == 'early_fusion':
        return _merged_loader(_append_split(plv, 'test'),
                              _append_split(pla, 'test'),
                              test_files, labels_at_tau,
                              fv=exp['feat_v'], fa=exp['feat_a'],
                              te=MAX_LEN_EARLY,
                              shuffle=False, permute_v=pv, permute_a=pa)
    # neural_concat / late_fusion use bimodal
    return _bi_loader(_append_split(plv, 'test'),
                      _append_split(pla, 'test'),
                      test_files, labels_at_tau,
                      fv=exp['feat_v'], fa=exp['feat_a'],
                      tv=MAX_LEN_V, ta=MAX_LEN_A,
                      shuffle=False, permute_v=pv, permute_a=pa)


def sweep_tau(exp_name):
    """Evaluate each trained model across TAU_GRID_MS.

    Only evaluates model kinds the experiment actually requests — reuse
    experiments (rows 6, 9-shuffled) inherit a full source model dict
    via Cell 35's aliasing, but we respect `exp['models']` here.

    """
    exp = EXPERIMENTS[exp_name]
    models = trained_models[exp_name]
    curves = {}

    requested = set(exp['models'])
    for kind, model in models.items():
        if kind.startswith('_'):
            continue  # skip private registry entries like _late_fusion_w
        if kind not in requested:
            continue
        curves[kind] = {}
        bimodal = kind in ('neural_concat',)
        for tau_ms in TAU_GRID_MS:
            labels_at_tau = LABELS_BY_TAU[tau_ms]
            loader = _test_loader_at_tau(exp, kind, tau_ms, labels_at_tau)
            name = f'{exp_name}_{kind}_tau{tau_ms:04d}'
            evaluate(model, loader, name, bimodal=bimodal)
            curves[kind][tau_ms] = results_table[name]

    # Late fusion per τ. Use the experiment's stored best_w for
    # consistency with run_experiment (reuse-mode gets the source's w
    # via Cell 35's dict copy; train-mode gets its own tuned w).
    if 'late_fusion' in requested \
            and 'unimodal_v' in models and 'unimodal_a' in models:
        curves['late_fusion'] = {}
        lf_w = models.get('_late_fusion_w', 0.5)
        for tau_ms in TAU_GRID_MS:
            labels_at_tau = LABELS_BY_TAU[tau_ms]
            l_v = _test_loader_at_tau(exp, 'unimodal_v', tau_ms, labels_at_tau)
            l_a = _test_loader_at_tau(exp, 'unimodal_a', tau_ms, labels_at_tau)
            pv, lab = get_probs(models['unimodal_v'], l_v)
            pa, _   = get_probs(models['unimodal_a'], l_a)
            preds = late_fusion_predict(pv, pa, w=lf_w)
            name = f'{exp_name}_late_fusion_tau{tau_ms:04d}'
            _store_late_fusion(name, preds, lab)
            curves['late_fusion'][tau_ms] = results_table[name]
    return curves


for name in EXPERIMENTS:
    print(f'\n>>> τ-sweep: {name}')
    tau_curves[name] = sweep_tau(name)


In [ ]:
def _n_at_tau(tau_ms):
    all_test = sorted(f for f in os.listdir(ENUM_DIR / 'test') if f.endswith('.npy'))
    labels_at_tau = LABELS_BY_TAU[tau_ms]
    return sum(1 for f in all_test if labels_at_tau.get(f[:-4], 999) in (0, 1, 2))

_n_per_tau = {t: _n_at_tau(t) for t in TAU_GRID_MS}
print(f'Test-set sizes per τ: {_n_per_tau}')


# Plot 1: macro-F1 × τ, one line per (experiment, model_kind).
fig, ax = plt.subplots(figsize=(11, 5))
for exp_name, by_kind in tau_curves.items():
    for kind, by_tau in by_kind.items():
        xs = sorted(by_tau.keys())
        ys = [by_tau[t]['macro_f1'] for t in xs]
        ax.plot(xs, ys, marker='o', label=f'{exp_name}/{kind}')
ax.set_xscale('log')
ax.set_xticks(TAU_GRID_MS)
# Annotate τ ticks with test-set size so differing-N comparisons are visible.
ax.set_xticklabels([f'{t}\n(n={_n_per_tau[t]})' for t in TAU_GRID_MS])
ax.set_xlabel('Prediction horizon τ (ms) — n = test samples classifiable as {0,1,2}')
ax.set_ylabel('Macro-F1 (%)')
ax.set_title('Horizon curve — macro-F1 vs. τ across all experiments')
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=8)
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


# Plot 2: per-class F1 × τ for the best NON-control (exp, kind).
# selected from the legitimate (non-permuted) experiments only.
def _pick_best():
    best_name, best_kind, best_f1 = None, None, -1
    for exp_name, by_kind in tau_curves.items():
        if 'permuted' in exp_name or 'shuffled' in exp_name:
            continue
        for kind, by_tau in by_kind.items():
            v = by_tau.get(TRAIN_TAU_MS, {}).get('macro_f1', -1)
            if v > best_f1:
                best_f1, best_name, best_kind = v, exp_name, kind
    return best_name, best_kind

pick_exp, pick_kind = _pick_best()
print(f'Per-class F1 curve for: {pick_exp} / {pick_kind}')
fig, ax = plt.subplots(figsize=(8, 4.5))
curve = tau_curves[pick_exp][pick_kind]
xs = sorted(curve.keys())
for ci, cname in enumerate(label_names):
    ys = [curve[t]['f1_per_class'][ci] for t in xs]
    ax.plot(xs, ys, marker='o', label=cname)
ax.set_xscale('log')
ax.set_xticks(TAU_GRID_MS)
ax.set_xticklabels([f'{t}\n(n={_n_per_tau[t]})' for t in TAU_GRID_MS])
ax.set_xlabel('Prediction horizon τ (ms) — n = test samples classifiable as {0,1,2}')
ax.set_ylabel('Per-class F1 (%)')
ax.set_title(f'Per-class horizon curve — {pick_exp} / {pick_kind}')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


---
## Section 10: Permuted-Dyad Control — Interpretation

Compare row 1 vs. row 6, and row 9 vs. row 9-shuffled. If performance at τ=400 ms drops when the listener's face (and audio, for row 9-shuffled) is replaced with a random other participant's stream, the model is using **dyadic coordination signal** — i.e., specifically the listener's real face matters, not just any face (spec §8.2).

If the drop is negligible, the "face helps turn-taking" result is **monadic** — the visual stream is informative about the speaker's own end-of-turn but doesn't carry dyadic information. That's still a scientifically valid outcome but needs different framing in the report.

In [ ]:
# Summary at training τ for quick eyeballing.
print('\n' + '=' * 80)
print(f'Summary @ τ={TRAIN_TAU_MS} ms')
print('=' * 80)
print(f'{"Experiment / Model":<50} {"Acc":>7} {"MacroF1":>8} {"H/Y/BC"}')
print('-' * 80)
for exp_name in EXPERIMENTS:
    if exp_name not in tau_curves:
        continue
    for kind, by_tau in tau_curves[exp_name].items():
        m = by_tau.get(TRAIN_TAU_MS)
        if m is None:
            continue
        per = m['f1_per_class']
        print(f'{exp_name+"/"+kind:<50} {m["acc"]:>6.2f}% {m["macro_f1"]:>7.2f} '
              f'[{per[0]:.0f}/{per[1]:.0f}/{per[2]:.0f}]')
print('=' * 80)

# Paired comparison: row 1 vs row 6 (permuted) at every τ, neural_concat only.
print('\nRow 1 vs. Row 6 (permuted) — neural_concat macro-F1 drop by τ:')
for t in TAU_GRID_MS:
    r1 = tau_curves['row1_dyadic_cross_pair']['neural_concat'][t]['macro_f1']
    r6 = tau_curves['row6_permuted_dyad_control']['neural_concat'][t]['macro_f1']
    print(f'  τ={t:>4} ms: row1={r1:5.2f} row6={r6:5.2f} Δ={r1-r6:+5.2f}')

print('\nRow 9 vs. Row 9-shuffled — neural_concat macro-F1 drop by τ:')
for t in TAU_GRID_MS:
    r9 = tau_curves['row9_mmvap_full_dyadic']['neural_concat'][t]['macro_f1']
    r9s = tau_curves['row9_shuffled_control']['neural_concat'][t]['macro_f1']
    print(f'  τ={t:>4} ms: row9={r9:5.2f} row9s={r9s:5.2f} Δ={r9-r9s:+5.2f}')
